# Notebook 1 – Download Market Data

This notebook downloads the price data used in the rest of the project and saves it to two CSV files:

- `data/adj_close.csv`: adjusted closing prices of the nine stocks
- `data/benchmark.csv`: adjusted closing prices of the benchmark

**Workflow:** import libraries → define the portfolio → download data → select adjusted closing prices → align dates and handle missing values → check the data → save the files

## 1. Import libraries

In [ ]:
import os
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Define the portfolio and the benchmark

The portfolio holds nine large eurozone companies, all listed on Euronext Paris, Euronext Amsterdam or Xetra (Frankfurt) and **traded in EUR**. The analysis takes the perspective of a euro-based investor, so no currency conversion is needed.

The benchmark is the **Xtrackers EURO STOXX 50 UCITS ETF 1C** (`XESC.DE`, Xetra, EUR). It is an accumulating fund: it reinvests the dividends it receives, and it tracks the EURO STOXX 50 Net Return index. All nine stocks are members of this index.

In [ ]:
tickers = [
    "ASML.AS",   # ASML Holding - Euronext Amsterdam
    "BNP.PA",    # BNP Paribas - Euronext Paris
    "MC.PA",     # LVMH - Euronext Paris
    "SIE.DE",    # Siemens - Xetra
    "AIR.PA",    # Airbus - Euronext Paris
    "OR.PA",     # L'Oréal - Euronext Paris
    "SAN.PA",    # Sanofi - Euronext Paris
    "SAP.DE",    # SAP - Xetra
    "ALV.DE"     # Allianz - Xetra
]

benchmark = "XESC.DE"   # Xtrackers EURO STOXX 50 UCITS ETF 1C

start_date = "2019-01-01"
end_date = "2025-12-31"   # the end date is not included, so the last possible date is 30 December 2025

## 3. Download the data

The nine stocks and the benchmark are downloaded together. `auto_adjust=False` keeps both the Close and the **Adjusted Close** prices.

In [ ]:
data = yf.download(tickers + [benchmark], start=start_date, end=end_date, auto_adjust=False)

data.head()

## 4. Select the Adjusted Close prices

The **Adjusted Close** price corrects past prices for dividends and stock splits. Returns calculated from it therefore include dividends (a total return), which is what an investor actually earned.

In [ ]:
prices = data["Adj Close"]

prices.head()

## 5. Align the dates and handle missing values

The stocks trade on three exchanges that share most, but not all, holidays. For example, Xetra is closed on 24 and 31 December while Euronext is open. On such a day the Xetra stocks and the benchmark have no price, so the table contains missing values.

The number of missing prices for each ticker is shown below.

In [ ]:
prices.isna().sum()

To keep the analysis simple and consistent, we keep only the **common trading dates**: the dates on which all nine stocks and the benchmark have a price. `dropna()` removes every other date.

- The stocks and the benchmark then cover exactly the same dates.
- No price is invented: we do not fill gaps or use any future information.
- After a removed date, the next daily return simply covers two days instead of one.

Only a few dates per year are removed, because the exchanges are closed on the same days most of the time.

In [ ]:
rows_before = len(prices)
prices = prices.dropna()

print("Dates before cleaning:", rows_before)
print("Dates after cleaning: ", len(prices))
print("Dates removed:        ", rows_before - len(prices))

In [ ]:
# Separate the stocks and the benchmark
adj_close = prices[tickers]
benchmark_prices = prices[[benchmark]]

## 6. Check the data

Before saving, we check:

- **Dimensions**: nine stock columns and one benchmark column, with the same number of rows.
- **Date range**: the data should start in early January 2019 and end in late December 2025. A later start or earlier end would mean that one ticker has missing history, which would shorten the whole sample.
- **Missing values**: none should remain.

In [ ]:
print("Stock prices:    ", adj_close.shape[0], "dates x", adj_close.shape[1], "stocks")
print("Benchmark prices:", benchmark_prices.shape[0], "dates x", benchmark_prices.shape[1], "benchmark")
print("Date range:      ", adj_close.index.min().date(), "to", adj_close.index.max().date())
print("Missing values remaining:", adj_close.isna().sum().sum() + benchmark_prices.isna().sum().sum())

In [ ]:
adj_close.head()

In [ ]:
benchmark_prices.head()

A quick chart is a useful visual check. Each price is rebased to 100 on the first date, so all series start at the same level. A sudden jump or a flat line would suggest a data problem.

In [ ]:
rebased = prices / prices.iloc[0] * 100

rebased.plot(figsize=(12, 6), title="Prices rebased to 100 (stocks and benchmark)")
plt.ylabel("Price (first date = 100)")
plt.show()

## 7. Save the data

The two files are saved in the `data` folder, which is created if it does not exist yet. Notebooks 2 and 3 and the dashboard read these files.

In [ ]:
os.makedirs("../data", exist_ok=True)

adj_close.to_csv("../data/adj_close.csv")
benchmark_prices.to_csv("../data/benchmark.csv")

print("Saved data/adj_close.csv and data/benchmark.csv")